# 2. Costed fire stations, written with an incidence matrix

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 6–8. The same model as demo 1, but
the constraints are built in a loop from a table of 0s and 1s. This is also the answer to task 2 on the
*Tutorial* slide (page 25).

**The problem.** Four fire stations $a, b, c, d$ can be run; each covers some of the cities 0–4 and has a
running cost (slides, PDF pages 6–8):

| Station | Covers cities | Cost |
| --- | --- | --- |
| $a$ | 0, 2 | 12 |
| $b$ | 1, 2, 3 | 7 |
| $c$ | 1, 4 | 10 |
| $d$ | 4 | 5 |

Choose stations so that every city is covered by at least one of them, at the lowest total cost. $x_t = 1$ if
station $t$ is run:

minimise $\sum_{t \in S} c_t x_t$ subject to $\sum_{t:\, e \in s_t} x_t \ge 1$ for every city $e$, $x_t \in \{0, 1\}$.

It is the minimum set cover of Week 05 (demo 3), with a cost on each set.

**The incidence matrix.** `A` has one row per city and one column per station; `A[city][station] = 1` if that
station covers that city. Row $e$ of `A` gives the constraint for city $e$:
$\sum_{t} A_{e t}\, x_t \ge 1$.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Cost **24**: stations $a$, $b$ and $d$ (`x[0]`, `x[1]`, `x[3]` = 1), the same as demo 1.

*Original script: `5 Source Files/Python/costed_firestation_incidence_matrix.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model and one binary variable per station (`x[0]` … `x[3]` = $a$ … $d$).

In [ ]:
# Create the model and one binary variable per station: x[0]..x[3] = stations a..d
model = ConcreteModel()
model.x = Var(range(4), within=Binary)

The data: the incidence matrix (rows = cities, columns = stations) and the costs. Its size gives the number of cities and stations, so nothing else is typed in by hand.

In [ ]:
# Define the incidence matrix: rows = cities 0-4, columns = stations a-d, 1 = station covers city
A = [
    [1, 0, 0, 0],   # city 0: a
    [0, 1, 1, 0],   # city 1: b, c
    [1, 1, 0, 0],   # city 2: a, b
    [0, 1, 0, 0],   # city 3: b
    [0, 0, 1, 1]    # city 4: c, d
]

no_cities = len(A)       # number of rows
no_stations = len(A[0])  # number of columns

c = [12, 7, 10, 5]   # running cost of stations a, b, c, d

The objective: the total running cost, minimised.

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(c[station] * model.x[station] for station in range(no_stations)), sense=minimize)

One constraint per row of `A`: the stations with a 1 in that row must cover the city at least once. The `0 * x` terms add nothing, so each constraint is the same as in demo 1.

In [ ]:
# Create a list of constraints based on the incidence matrix: one per city (row of A)
model.constraints = ConstraintList()
for city in range(no_cities):
    constraint_expr = sum(A[city][station] * model.x[station] for station in range(no_stations)) >= 1
    model.constraints.add(constraint_expr)

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model: the five constraints are the five rows of `A`.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the chosen stations (value 1) and the total cost.

In [ ]:
# Display the results
model.display()

## Try it

Station $d$'s running cost rises from 5 to 11. Change one number in the code and run all cells again. What is
the new plan?

<details><summary>Answer</summary>

Change `c = [12, 7, 10, 5]` to `c = [12, 7, 10, 11]`. Stations $a$ and $b$ are still forced (cities 0 and 3),
and city 4 is now cheaper to cover with station $c$: stations $a$, $b$, $c$ at a cost of $12 + 7 + 10 = 29$.
Only the data changed; the constraints, built from `A`, did not.
</details>